# XGBoost–SARIMA model experiment
Production-oriented hybrid experiment with calendar/exogenous features, lag-52, separate holiday blend weights, chronological validation, and a portable artifact for inference.

In [ ]:
%pip install -q "xgboost>=3,<4" "statsmodels>=0.14,<1" "wandb>=0.19,<1" "joblib>=1.4,<2"

In [ ]:
from pathlib import Path
import warnings, joblib, numpy as np, pandas as pd
from xgboost import XGBRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX
warnings.filterwarnings('ignore')
DATA_DIR=Path('/content/drive/MyDrive/walmart_competition_data') if Path('/content').exists() else Path('../../data')
OUTPUT_DIR=Path('/content/drive/MyDrive/walmart_models') if Path('/content').exists() else Path('artifacts'); OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
VALIDATION_WEEKS=39; ORDERS=[((1,0,1),(0,1,1,52)),((0,1,1),(0,1,1,52))]
train=pd.read_csv(DATA_DIR/'train.csv',parse_dates=['Date']); test=pd.read_csv(DATA_DIR/'test.csv',parse_dates=['Date'])
features=pd.read_csv(DATA_DIR/'features.csv',parse_dates=['Date']); stores=pd.read_csv(DATA_DIR/'stores.csv')
def merge_tables(df): return df.merge(features,on=['Store','Date','IsHoliday'],how='left').merge(stores,on='Store',how='left')
train=merge_tables(train); test=merge_tables(test)

In [ ]:
BASE=['Store','Dept','IsHoliday','Size','Temperature','Fuel_Price','CPI','Unemployment','MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']
def make_x(df,history=None):
    z=df.copy(); z['Type']=z.Type.map({'A':0,'B':1,'C':2}); z['year']=z.Date.dt.year; z['month']=z.Date.dt.month
    z['week']=z.Date.dt.isocalendar().week.astype(int); z['week_sin']=np.sin(2*np.pi*z.week/52); z['week_cos']=np.cos(2*np.pi*z.week/52)
    if history is not None:
        lag=history[['Store','Dept','Date','Weekly_Sales']].copy(); lag.Date=lag.Date+pd.Timedelta(weeks=52); lag=lag.rename(columns={'Weekly_Sales':'lag_52'})
        z=z.merge(lag,on=['Store','Dept','Date'],how='left')
    else: z['lag_52']=np.nan
    cols=BASE+['Type','year','month','week','week_sin','week_cos','lag_52']
    return z[cols].astype(float).fillna(-999),cols
def wmae(y,p,h): return np.average(np.abs(np.asarray(y)-p),weights=np.where(np.asarray(h),5.,1.))
dates=np.sort(train.Date.unique()); cut=dates[-VALIDATION_WEEKS]; tr=train[train.Date<cut].copy(); va=train[train.Date>=cut].copy()
Xtr,cols=make_x(tr,tr); Xva,_=make_x(va,tr)
model=XGBRegressor(n_estimators=1600,max_depth=9,min_child_weight=8,learning_rate=.025,subsample=.85,colsample_bytree=.8,reg_lambda=4,objective='reg:absoluteerror',tree_method='hist',random_state=42)
model.fit(Xtr,tr.Weekly_Sales,sample_weight=np.where(tr.IsHoliday,5.,1.)); px=model.predict(Xva)

In [ ]:
def sarima_predictions(history,future,order,seasonal):
    out=np.full(len(future),np.nan)
    for key,g in future.groupby(['Store','Dept'],sort=False):
        y=history[(history.Store==key[0])&(history.Dept==key[1])].sort_values('Date').Weekly_Sales
        if len(y)<80: continue
        try: out[future.index.get_indexer(g.index)]=SARIMAX(y,order=order,seasonal_order=seasonal,enforce_stationarity=False,enforce_invertibility=False).fit(disp=False,maxiter=60).forecast(len(g))
        except Exception: pass
    return out
best=None
for order,seasonal in ORDERS:
    ps=sarima_predictions(tr,va,order,seasonal); ps=np.where(np.isfinite(ps),ps,px)
    for normal_w in np.arange(.5,.91,.1):
      for holiday_w in np.arange(.65,.96,.1):
        weights=np.where(va.IsHoliday,holiday_w,normal_w); blend=weights*px+(1-weights)*ps; score=wmae(va.Weekly_Sales,blend,va.IsHoliday)
        if best is None or score<best['score']: best={'score':score,'order':order,'seasonal_order':seasonal,'normal_xgb_weight':float(normal_w),'holiday_xgb_weight':float(holiday_w)}
print(best)

In [ ]:
# Refit XGBoost on all labeled rows. Inference refits SARIMA from stored history.
Xall,cols=make_x(train,train); final_model=model.set_params(n_estimators=max(300,int(model.n_estimators*len(train)/len(tr))))
final_model.fit(Xall,train.Weekly_Sales,sample_weight=np.where(train.IsHoliday,5.,1.))
bundle={'xgb':final_model,'feature_columns':cols,'history':train[['Store','Dept','Date','Weekly_Sales']].copy(),'features_table':features,'stores_table':stores,**best}
path=OUTPUT_DIR/'xgboost_sarima_pipeline.joblib'; joblib.dump(bundle,path,compress=3); print(path)